<a href="https://colab.research.google.com/github/DattaSai03/CodeAlpha_AIML/blob/main/DiseasePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xgboost
!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [ ]:
heart_data = pd.read_csv("heart.csv")

diabetes_data = pd.read_csv("diabetes.csv")

cancer_data = pd.read_csv("data.csv")
columns_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for column in columns_to_replace:
    diabetes_data[column] = diabetes_data[column].replace(
        0,
        diabetes_data[column].median()
    )
    print((diabetes_data[['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']] == 0).sum())

In [ ]:
print(heart_data.head())

print(diabetes_data.head())

print(cancer_data.head())

In [ ]:
print(heart_data.info())

print(diabetes_data.info())

print(cancer_data.info())

In [ ]:
print(heart_data.isnull().sum())

print(diabetes_data.isnull().sum())

print(cancer_data.isnull().sum())

In [ ]:
if 'Unnamed: 32' in cancer_data.columns:
    cancer_data.drop('Unnamed: 32', axis=1, inplace=True)

if 'id' in cancer_data.columns:
    cancer_data.drop('id', axis=1, inplace=True)

if cancer_data['diagnosis'].dtype == 'object':
    cancer_data['diagnosis'] = cancer_data['diagnosis'].map({
        'M': 1,
        'B': 0
    })

In [ ]:
def train_and_evaluate(data, target_column, dataset_name):

    print("\n================================================")
    print(f"{dataset_name} DATASET")
    print("================================================")

    X = data.drop(target_column, axis=1)
    y = data[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
    smote = SMOTE(random_state=42)

    X_train, y_train = smote.fit_resample(X_train, y_train)

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "SVM": SVC(),
        "Random Forest": RandomForestClassifier(),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    }

    for name, model in models.items():

        print("\n--------------------------------------")
        print(f"MODEL: {name}")
        print("--------------------------------------")

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)

        precision = precision_score(y_test, y_pred)

        recall = recall_score(y_test, y_pred)

        f1 = f1_score(y_test, y_pred)

        print(f"Accuracy  : {accuracy:.4f}")
        print(f"Precision : {precision:.4f}")
        print(f"Recall    : {recall:.4f}")
        print(f"F1 Score  : {f1:.4f}")

        print("\nClassification Report")
        print(classification_report(y_test, y_pred))

        cm = confusion_matrix(y_test, y_pred)

        print("\nConfusion Matrix")
        print(cm)

        disp = ConfusionMatrixDisplay(confusion_matrix=cm)

        disp.plot()

        plt.title(f"{dataset_name} - {name}")

        plt.show()

In [ ]:
train_and_evaluate(
    heart_data,
    "target",
    "Heart Disease"
)

In [ ]:
train_and_evaluate(
    diabetes_data,
    "Outcome",
    "Diabetes"
)

In [ ]:
train_and_evaluate(
    cancer_data,
    "diagnosis",
    "Breast Cancer"
)